# 49W — Transcribe & Speaker Diarization
Uses **WhisperX** on GPU to transcribe audio and label speaker turns (SPEAKER_00, SPEAKER_01, ...).

**Runtime:** GPU (T4 or better). Runtime → Change runtime type → T4 GPU

**Input:** `MyDrive/49w-bot/data/audio/*.wav`

**Output:** `MyDrive/49w-bot/data/diarized/*.json` — transcript with speaker labels per turn

In [ ]:
!pip install -q whisperx
!pip install -q pyannote.audio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── CONFIG ─────────────────────────────────────────────────
AUDIO_DIR    = "/content/drive/MyDrive/49w-bot/data/audio"
OUTPUT_DIR   = "/content/drive/MyDrive/49w-bot/data/diarized"
VIDEOS_JSON  = "/content/drive/MyDrive/49w-bot/data/raw/videos.json"

# Get your free HuggingFace token at https://huggingface.co/settings/tokens
# Accept conditions at: https://huggingface.co/pyannote/speaker-diarization-3.1
HF_TOKEN = "hf_..."  # <-- paste your token here

WHISPER_MODEL   = "large-v3"   # large-v3 for best Turkish accuracy
LANGUAGE        = "tr"         # Turkish
NUM_SPEAKERS    = 3            # expected number of hosts
# ────────────────────────────────────────────────────────────

In [ ]:
import whisperx
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"
print(f"Using device: {device}")

# Load Whisper model
print(f"Loading WhisperX {WHISPER_MODEL}...")
model = whisperx.load_model(WHISPER_MODEL, device, compute_type=compute_type, language=LANGUAGE)

# Load diarization pipeline
print("Loading diarization pipeline...")
diarize_model = whisperx.DiarizationPipeline(use_auth_token=HF_TOKEN, device=device)
print("Models ready.")

In [ ]:
import json
from pathlib import Path
from tqdm.notebook import tqdm

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Load video metadata for titles
with open(VIDEOS_JSON, encoding="utf-8") as f:
    video_meta = {v["video_id"]: v for v in json.load(f)}

audio_files = sorted(Path(AUDIO_DIR).glob("*.wav"))
print(f"Found {len(audio_files)} audio files")

done, skipped, failed = 0, 0, 0

for audio_path in tqdm(audio_files, desc="Transcribing"):
    vid_id = audio_path.stem
    out_file = Path(OUTPUT_DIR) / f"{vid_id}.json"

    if out_file.exists():
        skipped += 1
        continue

    try:
        # 1. Transcribe with Whisper
        audio = whisperx.load_audio(str(audio_path))
        result = model.transcribe(audio, batch_size=16)

        # 2. Align word timestamps
        align_model, metadata = whisperx.load_align_model(language_code=LANGUAGE, device=device)
        result = whisperx.align(result["segments"], align_model, metadata, audio, device)

        # 3. Diarize (who spoke when)
        diarize_segments = diarize_model(audio, min_speakers=2, max_speakers=NUM_SPEAKERS)

        # 4. Assign speakers to words
        result = whisperx.assign_word_speakers(diarize_segments, result)

        # 5. Merge into speaker turns
        turns = []
        for seg in result["segments"]:
            speaker = seg.get("speaker", "UNKNOWN")
            text = seg["text"].strip()
            if not text:
                continue
            # Merge consecutive segments from same speaker
            if turns and turns[-1]["speaker"] == speaker:
                turns[-1]["text"] += " " + text
                turns[-1]["end"] = seg["end"]
            else:
                turns.append({"speaker": speaker, "start": seg["start"], "end": seg["end"], "text": text})

        meta = video_meta.get(vid_id, {})
        record = {
            "video_id": vid_id,
            "title": meta.get("title", ""),
            "turns": turns,
            "full_text": " ".join(t["text"] for t in turns),
            "speakers": list(set(t["speaker"] for t in turns)),
        }

        with open(out_file, "w", encoding="utf-8") as f:
            json.dump(record, f, ensure_ascii=False, indent=2)
        done += 1

    except Exception as e:
        print(f"  Failed {vid_id}: {e}")
        failed += 1

print(f"\nDone: {done} | Skipped: {skipped} | Failed: {failed}")